# Attribute Computation and Filtering

This notebook shows how to compute node attributes on weighted morphological trees and use those attributes to build simple connected filters. It uses the same image to compare a tree of shapes, a max-tree, and a min-tree.


In [ ]:
import mmcfilters
import math
from pathlib import Path

import cv2 as cv
from IPython import display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import mtviz as viz

plt.rcParams["figure.figsize"] = (16, 16)


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)


## Load and inspect the input image

The examples use a grayscale test image. The image values define the altitude of the component tree nodes, while the grid adjacency controls which pixels may belong to the same connected component.


In [ ]:
input_image = load_grayscale("../dat/imgTeste.png")
#input_image = load_grayscale("../dat/imgObjetos.png")
#input_image = load_grayscale("../dat/cable.png")
#input_image = load_grayscale("../dat/images/peppers_gray.tif")

(num_rows, num_cols) = input_image.shape



## Build a weighted tree of shapes and compute attributes

`computeAttributes` evaluates several attribute calculators in one pass and returns a table indexed by node. The notebook keeps the attribute names in `attribute_indices` so later cells can select columns without hard-coded positions.


In [ ]:
AT = mmcfilters.Attribute
tree = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.SelfDual)
mmcfilters.Attribute.computeAttributes(tree, [AT.AREA, AT.VOLUME, AT.BOX_HEIGHT])

In [ ]:
import pandas as pd
ok = True
selected_attributes = [
    mmcfilters.Attribute.AREA,
    mmcfilters.Attribute.INERTIA,
    mmcfilters.Attribute.BOX_HEIGHT,
]
trainset = ["../dat/imgTeste.png", "../dat/imgObjetos.png", "../dat/cable.png"]
for i in range(len(trainset)):
    img = load_grayscale(trainset[i])
    (img_rows, img_cols) = img.shape
    tree = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(img, interpolation=mmcfilters.ToSInterpolation.SelfDual)
    features, attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(tree, selected_attributes)

    df2 = pd.DataFrame(attribute_matrix, columns=list(features.keys()))
    if len(df2[df2.isna().any(axis=1)]) > 0:
        print(i, df2[df2.isna().any(axis=1)])
        ok = False
if ok:
    print("Attributes verified, no NaN values found in the selected subset.")

## Compare tree families

The same attribute set is computed on a max-tree, a min-tree, and a tree of shapes. This makes it easier to check whether an attribute behaves consistently across different tree topologies.


In [ ]:
max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(input_image, radius=1.5)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(input_image, radius=1.5)
tos = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.SelfDual)
tos_4c8c = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.Min4cMax8c)

max_tree_filter = mmcfilters.AttributeFilters(max_tree)
min_tree_filter = mmcfilters.AttributeFilters(min_tree)
tos_filter = mmcfilters.AttributeFilters(tos)
tos_4c8c_filter = mmcfilters.AttributeFilters(tos_4c8c)

selected_attributes = [
    mmcfilters.Attribute.AREA,
    mmcfilters.Attribute.INERTIA,
    mmcfilters.Attribute.BOX_HEIGHT,
]
attribute_indices, max_tree_attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(max_tree, selected_attributes)
attribute_indices, min_tree_attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(min_tree, selected_attributes)
attribute_indices, tos_attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(tos, selected_attributes)
attribute_indices, tos_4c8c_attribute_matrix = mmcfilters.Attribute.computeTopologyAttributes(tos_4c8c, selected_attributes)
attribute_indices

## Inspect nodes and contours

The tree printer summarizes the parent-child structure and the selected node attributes. The contour helper below is used to visualize which image regions are affected by later filters.


In [ ]:
pt = viz.PrintTree(lambda n: tos.getChildren(n), lambda n: f"id:{n}: altitude: {tos.getAltitude(n)}")
pt(tos.getRoot())

In [ ]:
def plot_contours(pixels, node_id, num_rows, num_cols):
    contour_image = np.zeros((num_rows, num_cols), dtype=np.uint8)

    for p in pixels:
        row = p // num_cols
        col = p % num_cols
        contour_image[row, col] = 255

    plt.figure(figsize=(5, 5))
    plt.imshow(contour_image, cmap='Reds')
    plt.title(f"node {node_id}")
    plt.axis('off')
    plt.show()

tree_contours = mmcfilters.ContourComputation.extraction(max_tree)
area_col = attribute_indices['AREA']
for nodeId, contour in tree_contours.contoursByNode():
    if max_tree_attribute_matrix[nodeId, area_col] > 100:
        plot_contours(contour, nodeId, num_rows, num_cols)

## Filter by individual attributes

The following cells apply attribute thresholds directly to the tree. Area and inertia highlight different structures, so comparing their outputs helps validate both the attribute values and the reconstruction step.


In [ ]:
area_col = attribute_indices['AREA']
threshold = (num_rows*num_cols) *0.25

max_tree_area = max_tree_attribute_matrix[:, area_col]
min_tree_area = min_tree_attribute_matrix[:, area_col]
tos_area = tos_attribute_matrix[:, area_col]

max_tree_filtered_image = max_tree_filter.filteringDirectRule(max_tree_area > threshold)
min_tree_filtered_image = min_tree_filter.filteringDirectRule(min_tree_area > threshold)
tos_filtered_image = tos_filter.filteringDirectRule(tos_area > threshold)

plt.subplot(1,4, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,4, 2)
plt.imshow(max_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (max_tree)')

plt.subplot(1,4, 3)
plt.imshow(min_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (min_tree)')

plt.subplot(1,4, 4)
plt.imshow(tos_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (tos)')

In [ ]:
inertia_col = attribute_indices['INERTIA']
threshold = 0.2

max_tree_inertia = max_tree_attribute_matrix[:, inertia_col]
min_tree_inertia = min_tree_attribute_matrix[:, inertia_col]
tos_inertia = tos_attribute_matrix[:, inertia_col]

max_tree_filtered_image = max_tree_filter.filteringSubtractiveRule(max_tree_inertia > threshold)
min_tree_filtered_image = min_tree_filter.filteringSubtractiveRule(min_tree_inertia > threshold)
tos_filtered_image = tos_filter.filteringSubtractiveRule(tos_inertia > threshold)

plt.subplot(1,4, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,4, 2)
plt.imshow(max_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (max_tree)')

plt.subplot(1,4, 3)
plt.imshow(min_tree_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (min_tree)')

plt.subplot(1,4, 4)
plt.imshow(tos_filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter (tos)')